In [3]:
# Shared versions with speaker_diarization.ipynb. Restart the kernel after installation.
%pip install "torch==2.7.1+cu128" "torchvision==0.22.1+cu128" "torchaudio==2.7.1+cu128" --index-url https://download.pytorch.org/whl/cu128
%pip install "numpy==2.5.4" "huggingface-hub==0.34.4" "transformers==4.55.4" "datasets==4.0.0" "accelerate==1.15.0"
%pip check

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
No broken requirements found.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import shutil

os.environ["PATH"] += os.pathsep + r"C:\LUNA\ffmpeg-2026-10-15-git-afead44165-full_build\bin"

print(shutil.which("ffmpeg"))

C:\LUNA\ffmpeg-2026-10-15-git-afead44165-full_build\bin\ffmpeg.EXE


In [3]:
import os
import shutil
import subprocess

ffmpeg_dir = r"C:\LUNA\ffmpeg-2026-10-15-git-afead44165-full_build\bin"
os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ.get("PATH", "")
ffmpeg_exe = shutil.which("ffmpeg")
if not ffmpeg_exe:
    raise FileNotFoundError(f"FFmpeg not found in {ffmpeg_dir}")
subprocess.run([ffmpeg_exe, "-version"], check=True, capture_output=True)
print(f"FFmpeg: {ffmpeg_exe}")

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
# from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True, # 청크별로 타임스탬프를 반환
    chunk_length_s=10,      # 입력 오디오를 10초씩 나누기
    stride_length_s=2,      # 청크가 2초씩 겹치도록 나누기
)

# dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
# sample = dataset[0]["audio"]
sample = r"C:\LUNA\whisper\audio\Forward_Base_Taking_a_Day_Off.mp3"

result = pipe(sample)
print(result["text"])

FFmpeg: C:\LUNA\ffmpeg-2026-10-15-git-afead44165-full_build\bin\ffmpeg.EXE


Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/2

 Tadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadadad Every day than night, no more days to cry Longing for supreme supreme, peaceful daily life Put on my lips, I gr

In [ ]:
start_end_text = []

for chunk in result["chunks"]:
    start_end_text.append(
        {
            "start": chunk["timestamp"][0],
            "end": chunk["timestamp"][1],
            "text": chunk["text"],
        }
    )

import pandas as pd
df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
df.to_csv("Forward_Base_Taking_a_Day_Off.csv", index=False, sep="|", encoding="utf-8-sig")
display(df)

NameError: name 'result' is not defined